# HybridGraphSAGE Document Classification on RVL-CDIP Small-200

This notebook implements the hybrid CNN+GNN fusion approach:
- **CNN path**: ResNet-50 avgpool features (2048-dim global context)
- **GNN path**: GraphSAGE on 7x7 spatial feature maps with 2D positional encoding (2050-dim nodes)
- **Fusion**: Concatenation of GNN embedding (128-dim) + CNN global (2048-dim) → Linear(2176, 16)

**Inputs:** `cached_features/` (from Notebook 1, includes both layer4 and avgpool), `checkpoints/resnet50_baseline_metrics.json`, `checkpoints/graphsage_metrics.json`

**Outputs:** `checkpoints/hybrid_graphsage_best.pt`, `checkpoints/hybrid_graphsage_metrics.json`

---

## 1. Setup

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import sys
sys.path.insert(0, "..")

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
from torch_geometric.loader import DataLoader as PyGDataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from tqdm import tqdm

from src.config import Config
from src.data import load_rvl_cdip, RVL_CDIP_LABELS
from src.features import load_cached_features
from src.graph import build_grid_edge_index, build_graph_dataset_hybrid
from src.model import HybridGraphSAGE

print(f"PyTorch: {torch.__version__}")
print(f"PyG: {torch_geometric.__version__}")

In [ ]:
config = Config()
config.seed_everything()

CHECKPOINT_DIR = Path("../checkpoints")

# Load dataset metadata to populate RVL_CDIP_LABELS
_ = load_rvl_cdip(config)

print(f"\nLabels loaded: {len(RVL_CDIP_LABELS)} classes")
print(f"Device: {config.device}")

## 2. Load Cached Features

Load the pre-extracted ResNet-50 features from Notebook 1. Each cached file contains:
- **features** [2048, 7, 7]: layer4 spatial features (used as graph nodes)
- **global_feat** [2048]: avgpool global features (used in fusion classifier)
- **label**: document class

In [ ]:
cached_train = load_cached_features(config, "train")
cached_val = load_cached_features(config, "validation")

print(f"Train: {len(cached_train)} samples")
print(f"Val:   {len(cached_val)} samples")
print(f"\nSample structure:")
print(f"  features:    {cached_train[0]['features'].shape}")
print(f"  global_feat: {cached_train[0]['global_feat'].shape}")
print(f"  label:       {cached_train[0]['label']} ({RVL_CDIP_LABELS[cached_train[0]['label']]})")

## 3. Graph Construction with Positional Encoding

Convert each feature map to a graph with:
- **49 nodes** (7x7 grid) with **2050-dim** features (2048 CNN channels + 2 normalized position coords)
- **k=8 edges** (Moore neighborhood)
- **global_feat** stored as custom attribute [1, 2048] for correct PyG batching

In [ ]:
GRID_H, GRID_W = 7, 7
K_NEIGHBORS = 8

edge_index = build_grid_edge_index(GRID_H, GRID_W, k=K_NEIGHBORS)
print(f"Grid: {GRID_H}x{GRID_W} = {GRID_H * GRID_W} nodes")
print(f"k={K_NEIGHBORS} (Moore neighborhood), {edge_index.shape[1]} directed edges")

train_graphs = build_graph_dataset_hybrid(cached_train, edge_index)
val_graphs = build_graph_dataset_hybrid(cached_val, edge_index)
print(f"\nTrain graphs: {len(train_graphs)}")
print(f"Val graphs:   {len(val_graphs)}")

In [ ]:
g = train_graphs[0]
print(f"Node features: {g.x.shape}")
print(f"Edge index:    {g.edge_index.shape}")
print(f"Label:         {g.y.item()} ({RVL_CDIP_LABELS[g.y.item()]})")
print(f"Global feat:   {g.global_feat.shape}")

print(f"\nPositional encoding verification:")
print(f"  Node 0  (row=0, col=0): PE = {g.x[0, -2:].tolist()}")
print(f"  Node 6  (row=0, col=6): PE = {g.x[6, -2:].tolist()}")
print(f"  Node 42 (row=6, col=0): PE = {g.x[42, -2:].tolist()}")
print(f"  Node 48 (row=6, col=6): PE = {g.x[48, -2:].tolist()}")

In [ ]:
hybrid_train_loader = PyGDataLoader(train_graphs, batch_size=config.batch_size, shuffle=True)
hybrid_val_loader = PyGDataLoader(val_graphs, batch_size=config.batch_size, shuffle=False)

print(f"Train: {len(hybrid_train_loader)} batches")
print(f"Val:   {len(hybrid_val_loader)} batches")

batch = next(iter(hybrid_train_loader))
print(f"\nBatch structure:")
print(f"  x:           {batch.x.shape}")
print(f"  edge_index:  {batch.edge_index.shape}")
print(f"  y:           {batch.y.shape}")
print(f"  batch:       {batch.batch.shape} ({batch.batch.max().item() + 1} graphs)")
print(f"  global_feat: {batch.global_feat.shape}")

## 4. HybridGraphSAGE Model

Architecture with CNN+GNN fusion:

```
GNN path:  SAGEConv(2050, 256) -> ReLU -> Dropout(0.5) -> SAGEConv(256, 128) -> ReLU -> global_mean_pool -> [B, 128]
CNN path:  avgpool features -> [B, 2048]
Fusion:    concat([128] + [2048]) -> Linear(2176, 16) -> logits
```

In [ ]:
hybrid_model = HybridGraphSAGE(
    node_dim=2050,
    hidden_channels=256,
    embed_channels=128,
    global_channels=2048,
    num_classes=len(RVL_CDIP_LABELS),
    dropout=0.5,
).to(config.device)

total_params = sum(p.numel() for p in hybrid_model.parameters())
trainable_params = sum(p.numel() for p in hybrid_model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(hybrid_model)

## 5. Training

Using Adam with weight_decay=1e-4 for regularization on small dataset. Early stopping with patience=10.

In [ ]:
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 10

optimizer = torch.optim.Adam(
    hybrid_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
criterion = nn.CrossEntropyLoss()

print(f"Epochs: {NUM_EPOCHS}")
print(f"Optimizer: Adam (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
print(f"Early stopping patience: {PATIENCE}")

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0.0
patience_counter = 0
best_checkpoint_path = CHECKPOINT_DIR / "hybrid_graphsage_best.pt"

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    hybrid_model.train()
    train_loss_sum, train_correct, train_total = 0.0, 0, 0

    for batch in hybrid_train_loader:
        batch = batch.to(config.device)
        optimizer.zero_grad()
        logits = hybrid_model(batch.x, batch.edge_index, batch.batch, batch.global_feat)
        loss = criterion(logits, batch.y)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * batch.y.size(0)
        train_correct += (logits.argmax(dim=1) == batch.y).sum().item()
        train_total += batch.y.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    # --- Validate ---
    hybrid_model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for batch in hybrid_val_loader:
            batch = batch.to(config.device)
            logits = hybrid_model(batch.x, batch.edge_index, batch.batch, batch.global_feat)
            loss = criterion(logits, batch.y)

            val_loss_sum += loss.item() * batch.y.size(0)
            val_correct += (logits.argmax(dim=1) == batch.y).sum().item()
            val_total += batch.y.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    marker = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(hybrid_model.state_dict(), best_checkpoint_path)
        marker = " *"
    else:
        patience_counter += 1

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(
            f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
            f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}{marker}"
        )

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\nBest validation accuracy: {best_val_acc:.4f}")
print(f"Checkpoint: {best_checkpoint_path}")

In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, history["train_loss"], label="Train")
ax1.plot(epochs_range, history["val_loss"], label="Validation")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("HybridGraphSAGE — Training & Validation Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history["train_acc"], label="Train")
ax2.plot(epochs_range, history["val_acc"], label="Validation")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("HybridGraphSAGE — Training & Validation Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Evaluation

In [ ]:
hybrid_model.load_state_dict(torch.load(best_checkpoint_path, weights_only=True))
hybrid_model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in hybrid_val_loader:
        batch = batch.to(config.device)
        logits = hybrid_model(batch.x, batch.edge_index, batch.batch, batch.global_feat)
        preds = logits.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(batch.y.cpu().tolist())

hybrid_accuracy = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"HybridGraphSAGE validation accuracy: {hybrid_accuracy:.4f}")

In [ ]:
report = classification_report(
    all_labels,
    all_preds,
    target_names=RVL_CDIP_LABELS,
    digits=4,
)
print(report)

In [ ]:
hybrid_cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(hybrid_cm, interpolation="nearest", cmap="Blues")
ax.set_title("HybridGraphSAGE — Confusion Matrix")
plt.colorbar(im, ax=ax, fraction=0.046)

tick_marks = np.arange(len(RVL_CDIP_LABELS))
ax.set_xticks(tick_marks)
ax.set_xticklabels(RVL_CDIP_LABELS, rotation=45, ha="right", fontsize=8)
ax.set_yticks(tick_marks)
ax.set_yticklabels(RVL_CDIP_LABELS, fontsize=8)

thresh = hybrid_cm.max() / 2.0
for i in range(hybrid_cm.shape[0]):
    for j in range(hybrid_cm.shape[1]):
        ax.text(
            j, i, str(hybrid_cm[i, j]),
            ha="center", va="center",
            color="white" if hybrid_cm[i, j] > thresh else "black",
            fontsize=7,
        )

ax.set_ylabel("True Label")
ax.set_xlabel("Predicted Label")
plt.tight_layout()
plt.show()

## 7. Comparison: Hybrid vs Plain GraphSAGE vs CNN Baseline

In [ ]:
baseline_metrics_path = CHECKPOINT_DIR / "resnet50_baseline_metrics.json"
graphsage_metrics_path = CHECKPOINT_DIR / "graphsage_metrics.json"

with open(baseline_metrics_path) as f:
    baseline_metrics = json.load(f)
with open(graphsage_metrics_path) as f:
    graphsage_metrics = json.load(f)

hybrid_report_dict = classification_report(
    all_labels, all_preds, target_names=RVL_CDIP_LABELS, output_dict=True
)
hybrid_macro_f1 = f1_score(all_labels, all_preds, average="macro")

print(f"{'Model':<25} {'Val Acc':>10} {'Macro F1':>10}")
print("-" * 47)
print(f"{'CNN Baseline (ResNet-50)':<25} {baseline_metrics['accuracy']:>10.4f} {baseline_metrics['macro_avg']['f1']:>10.4f}")
print(f"{'Plain GraphSAGE':<25} {graphsage_metrics['accuracy']:>10.4f} {graphsage_metrics['macro_avg']['f1']:>10.4f}")
print(f"{'HybridGraphSAGE':<25} {hybrid_accuracy:>10.4f} {hybrid_macro_f1:>10.4f}")

In [ ]:
print(f"{'Class':<25} {'Baseline F1':>12} {'GraphSAGE F1':>13} {'Hybrid F1':>10} {'Delta (H-B)':>12}")
print("-" * 74)

for label in RVL_CDIP_LABELS:
    b_f1 = baseline_metrics["per_class"][label]["f1"]
    g_f1 = graphsage_metrics["per_class"][label]["f1"]
    h_f1 = hybrid_report_dict[label]["f1-score"]
    delta = h_f1 - b_f1
    print(f"{label:<25} {b_f1:>12.4f} {g_f1:>13.4f} {h_f1:>10.4f} {delta:>+12.4f}")

print("-" * 74)
b_macro = baseline_metrics["macro_avg"]["f1"]
g_macro = graphsage_metrics["macro_avg"]["f1"]
print(f"{'Macro Average':<25} {b_macro:>12.4f} {g_macro:>13.4f} {hybrid_macro_f1:>10.4f} {hybrid_macro_f1 - b_macro:>+12.4f}")

In [ ]:
baseline_cm = np.array(baseline_metrics["confusion_matrix"])
graphsage_cm = np.array(graphsage_metrics["confusion_matrix"])

fig, axes = plt.subplots(1, 3, figsize=(30, 9))

for ax, cm_data, title in [
    (axes[0], baseline_cm, "ResNet-50 Baseline"),
    (axes[1], graphsage_cm, "Plain GraphSAGE"),
    (axes[2], hybrid_cm, "HybridGraphSAGE"),
]:
    im = ax.imshow(cm_data, interpolation="nearest", cmap="Blues")
    ax.set_title(title, fontsize=12)
    plt.colorbar(im, ax=ax, fraction=0.046)

    tick_marks = np.arange(len(RVL_CDIP_LABELS))
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(RVL_CDIP_LABELS, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(RVL_CDIP_LABELS, fontsize=7)

    thresh = cm_data.max() / 2.0
    for i in range(cm_data.shape[0]):
        for j in range(cm_data.shape[1]):
            ax.text(
                j, i, str(cm_data[i, j]),
                ha="center", va="center",
                color="white" if cm_data[i, j] > thresh else "black",
                fontsize=6,
            )

    ax.set_ylabel("True Label")
    ax.set_xlabel("Predicted Label")

plt.suptitle("Confusion Matrix Comparison", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
hybrid_metrics = {
    "model": "hybrid_graphsage",
    "accuracy": hybrid_accuracy,
    "per_class": {
        label: {
            "precision": hybrid_report_dict[label]["precision"],
            "recall": hybrid_report_dict[label]["recall"],
            "f1": hybrid_report_dict[label]["f1-score"],
            "support": hybrid_report_dict[label]["support"],
        }
        for label in RVL_CDIP_LABELS
    },
    "macro_avg": {
        "precision": hybrid_report_dict["macro avg"]["precision"],
        "recall": hybrid_report_dict["macro avg"]["recall"],
        "f1": hybrid_report_dict["macro avg"]["f1-score"],
    },
    "confusion_matrix": hybrid_cm.tolist(),
    "history": history,
    "architecture": {
        "node_dim": 2050,
        "hidden_channels": 256,
        "embed_channels": 128,
        "global_channels": 2048,
        "fusion_dim": 2176,
        "dropout": 0.5,
    },
    "graph_config": {
        "grid": f"{GRID_H}x{GRID_W}",
        "k_neighbors": K_NEIGHBORS,
        "positional_encoding": "2d_normalized",
    },
}

hybrid_metrics_path = CHECKPOINT_DIR / "hybrid_graphsage_metrics.json"
with open(hybrid_metrics_path, "w") as f:
    json.dump(hybrid_metrics, f, indent=2)

print(f"Metrics saved to: {hybrid_metrics_path}")

## Summary

| Model | Val Accuracy | Macro F1 | Fusion | Parameters |
|-------|-------------|----------|--------|------------|
| CNN Baseline (ResNet-50) | see above | see above | N/A | ~32K trainable |
| Plain GraphSAGE | see above | see above | N/A | ~1.1M |
| HybridGraphSAGE | see above | see above | GNN[128] + CNN[2048] → Linear(2176, 16) | ~1.6M |

**Key observations:**
- HybridGraphSAGE combines local spatial reasoning (GNN on 7x7 grid) with global context (ResNet-50 avgpool)
- 2D positional encoding gives the GNN explicit spatial awareness of where each node sits in the feature map
- The fusion layer allows the classifier to leverage both fine-grained spatial patterns and holistic document features

**Next steps:**
- Compare additional GNN architectures (GCN, GAT) with the same hybrid fusion approach
- Ablation studies: with/without PE, different k values, projection layer vs direct concat
- Evaluation on RVL-CDIP-N (out-of-distribution)